# 00 · Common library (do not run directly)
**Project:** Cross-Lingual Generalization and Robustness of Multilingual AI Text Detectors (TMLR study)

This notebook is the project's **single shared library**. Every other notebook loads it with `load_common()` (first cell of each notebook), which executes its code cells in the caller's namespace.

It contains: paths & config loading, seeding, logging, run registry, I/O helpers, text normalisation, language ID, metrics, bootstrap/statistical tests.

**Rules:** no side effects at load time except reading the config and creating missing folders. Heavy models are loaded lazily.

In [ ]:
import os, sys, json, time, math, random, hashlib, logging, platform, subprocess, re, unicodedata, gc
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("MATD_ROOT", "/content/drive/MyDrive/multilingual-ai-text-detection"))
CONFIG_PATH = PROJECT_ROOT / "00_admin" / "config" / "config.json"

def load_config(path=CONFIG_PATH):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

CFG = load_config()

# Config migrations (applied in order, in place; the previous version is archived). See the project decision log.
# v1.1 (2026-10-10): dataset rebuild at 200 pairs per (language x generator) cell after the first data audit.
# v1.2 (2026-10-10): second audit — symmetric removal of parenthetical/bracketed spans (Wikipedia-lead metadata:
#   transliterations, birth dates; present in 67-88% of human vs 16-27% of machine texts), min_words 50 -> 40.
CONFIG_MIGRATIONS = [
    ("1.1", {"n_topics_target": 1000, "gen_oversample": 1.25, "gen_margin": 1.15, "wiki_min_bytes": 2500,
             "template_df": 5, "template_max_share": 0.3, "length_tol_words": 10, "length_tol_frac": 0.25,
             "max_topup_rounds": 4}),
    ("1.2", {"min_words": 40, "strip_parentheticals": True}),
]
_ver = lambda v: tuple(int(x) for x in str(v).split("."))

def migrate_config():
    """Upgrade config.json to the latest version (idempotent). Returns the reloaded config."""
    global CFG
    cfg = load_config()
    if _ver(cfg["config_version"]) < _ver(CONFIG_MIGRATIONS[-1][0]):
        archive = CONFIG_PATH.parent / f"config_v{cfg['config_version']}_{datetime.now():%Y%m%d%H%M%S}.json"
        archive.write_text(json.dumps(cfg, indent=1, ensure_ascii=False), encoding="utf-8")
        for v, upd in CONFIG_MIGRATIONS:
            if _ver(cfg["config_version"]) < _ver(v):
                cfg["data"].update(upd); cfg["config_version"] = v
        CONFIG_PATH.write_text(json.dumps(cfg, indent=1, ensure_ascii=False), encoding="utf-8")
        print(f"config migrated to v{cfg['config_version']} (previous version archived as {archive.name})")
    CFG = load_config()
    return CFG

# ---- folder layout (single source of truth: config["paths"]) ----
P = {k: PROJECT_ROOT / v for k, v in CFG["paths"].items()}
for _p in P.values():
    _p.mkdir(parents=True, exist_ok=True)

LANGS = list(CFG["languages"].keys())
LANG_META = CFG["languages"]
GENERATORS = CFG["generators"]                 # dict: gen_id -> {model, family, ...}
FAMILIES = sorted({g["family"] for g in GENERATORS.values()})
SEEDS = CFG["seeds"]

def utcnow():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

## Seeding & logging

In [ ]:
def set_seed(seed: int):
    """Seed python, numpy and (if imported) keras / torch."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    if "keras" in sys.modules:
        import keras
        keras.utils.set_random_seed(seed)
    if "torch" in sys.modules:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

def get_logger(notebook_name: str):
    log_file = P["logs"] / f"{notebook_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    logger = logging.getLogger(notebook_name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    for h in (logging.FileHandler(log_file, encoding="utf-8"), logging.StreamHandler(sys.stdout)):
        h.setFormatter(fmt); logger.addHandler(h)
    logger.info(f"log file: {log_file}")
    return logger

def env_snapshot(notebook_name: str):
    """Save python/package/GPU versions for reproducibility; returns dict."""
    info = {"notebook": notebook_name, "time_utc": utcnow(), "python": sys.version,
            "platform": platform.platform(), "config_version": CFG["config_version"]}
    try:
        info["gpu"] = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    except Exception:
        info["gpu"] = "none"
    try:
        info["pip_freeze"] = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                                            capture_output=True, text=True).stdout.splitlines()
    except Exception:
        info["pip_freeze"] = []
    out = P["env"] / f"env_{notebook_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    out.write_text(json.dumps(info, indent=1, ensure_ascii=False))
    return info

## I/O helpers

In [ ]:
def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def append_jsonl(path, rows):
    with open(path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def write_json(path, obj):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)

def save_parquet(df, path):
    tmp = Path(str(path) + ".tmp")
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def stable_hash(s: str, n=8):
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:n]

## Text normalisation (applied identically to human and machine text)
Removes surface cues that act as label shortcuts rather than linguistic evidence:
- extraction artefacts (double spaces, space before punctuation) — found in COLING-kk human texts;
- LLM formatting (markdown, a separate heading/title line, "Here is a paragraph:" preambles);
- typography that differs by source, not by author type: dash variants (Kazakh Wikipedia "—" vs LLM "–"),
  quote variants, and apostrophe variants (Uzbek Wikipedia oʻ/gʻ with U+02BB vs LLM ' or ‘);
- (v1.2) LLM italics (*x*), trailing notes after a horizontal rule ("---  *Note: ...*"), Wikipedia stress marks
  (combining acute/grave, e.g. Russian leads), and — if `strip_parentheticals` — parenthetical/bracketed spans
  (Wikipedia-lead metadata: transliterations, native names, birth/death dates), removed from BOTH classes.
Data audit (notebook 04) verifies that these cues no longer separate the classes.

In [ ]:
_ZW = dict.fromkeys([0x200B, 0x200C, 0x200D, 0x2060, 0xFEFF], None)   # zero-width chars
_TYPO = {**{c: 0x2014 for c in (0x2012, 0x2013, 0x2014, 0x2015)},       # figure/en/em dash, horizontal bar -> em dash
         0x2212: ord("-"),                                                # minus sign -> hyphen-minus
         **{c: ord('"') for c in (0x00AB, 0x00BB, 0x201E, 0x201C, 0x201D, 0x201F, 0xFF02)},   # « » „ “ ” ‟ ＂
         **{c: ord("'") for c in (0x02BB, 0x02BC, 0x2018, 0x2019, 0x201B, 0x0060, 0x00B4, 0x02B9, 0x2032)}}  # ʻ ʼ ‘ ’ ‛ ` ´ ʹ ′
_MD = [(re.compile(r"\*\*(.+?)\*\*"), r"\1"), (re.compile(r"__(.+?)__"), r"\1"),
       (re.compile(r"(?m)^\s{0,3}#{1,6}\s*"), ""), (re.compile(r"(?m)^\s*[-*•]\s+"), ""),
       (re.compile(r"(?<![*\w])\*(?!\s)([^*\n]+?)\*(?!\*)"), r"\1"),           # *italic*
       (re.compile(r"(?<!\w)_(?!\s)([^_\n]+?)_(?!\w)"), r"\1")]                   # _italic_
_HR = re.compile(r"(?:^|\s)(?:-{3,}|\*{3,}|_{3,})(?:\s|$)")                        # horizontal rule: text after it is an LLM note
_STRESS = re.compile("[\u0300\u0301]")                                         # combining grave/acute left after NFC = stress marks
_PAREN = re.compile(r"\s*\([^()]*\)"); _BRACK = re.compile(r"\s*\[[^\[\]]*\]")

def _strip_heading_lines(t: str) -> str:
    """Drop leading title/heading/preamble lines of multi-line text (single-line text is untouched)."""
    lines = [l.strip() for l in t.split("\n") if l.strip()]
    while len(lines) > 1:
        first = re.sub(r"[*#_]", "", lines[0]).strip()
        short_title = len(first.split()) <= 12 and not re.search(r"[.!?…]$", first)
        preamble = first.endswith(":") and len(first.split()) <= 20
        if short_title or preamble:
            lines.pop(0)
        else:
            break
    return "\n".join(lines)

def normalize_text(t: str) -> str:
    """Canonical surface form: NFC, no headings/markdown, unified typography, single spaces, one paragraph."""
    if t is None:
        return ""
    t = unicodedata.normalize("NFC", t).translate(_ZW)
    t = _STRESS.sub("", t)
    t = next((x for x in _HR.split(t) if x.strip()), "")   # keep the first non-empty block before a rule
    t = _strip_heading_lines(t)
    for pat, rep in _MD:
        t = pat.sub(rep, t)
    t = t.replace("*", "")
    t = t.translate(_TYPO)
    t = re.sub(r"\s+", " ", t)                       # newlines/tabs/multi-spaces -> single space
    if CFG["data"].get("strip_parentheticals", False):
        for _ in range(3):                           # nested parentheses
            t = _PAREN.sub("", t)
        t = _BRACK.sub("", t)
    t = re.sub(r"\s+([.,;:!?)\]%])", r"\1", t)      # no space before closing punctuation
    t = re.sub(r"([(\[])\s+", r"\1", t)             # no space after opening brackets
    t = re.sub(r"\s*—\s*", " — ", t)                 # one spacing convention for dashes
    return t.strip().strip('"').strip()

_SENT_END = re.compile(r"(?<=[.!?…])\s+")

def truncate_words(t: str, max_words: int) -> str:
    """Truncate to <= max_words, cutting at the last sentence boundary when possible."""
    words = t.split()
    if len(words) <= max_words:
        return t
    cut = " ".join(words[:max_words])
    sents = _SENT_END.split(cut)
    if len(sents) > 1:
        return " ".join(sents[:-1]).strip()
    return cut

def n_words(t: str) -> int:
    return len(t.split())

## Character-level transformations (shared by 05 attacks and 06 augmentation) and the Unicode defence
T3 homoglyph: replace a fraction `p` of eligible characters with look-alikes from the *other* script (Cyrillic↔Latin).
T4 typo: corrupt a fraction `p` of words (len ≥ 3) by swap / delete / duplicate / substitute.
Both are deterministic given (text, seed).

In [ ]:
_CYR2LAT = {"а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "у": "y", "х": "x", "і": "i", "ј": "j", "һ": "h", "ѕ": "s",
            "А": "A", "В": "B", "С": "C", "Е": "E", "Н": "H", "І": "I", "Ј": "J", "К": "K", "М": "M", "О": "O",
            "Р": "P", "Ѕ": "S", "Т": "T", "Х": "X", "Ү": "Y"}
_LAT2CYR = {v: k for k, v in _CYR2LAT.items()}

def _is_cyr(ch):
    return "\u0400" <= ch <= "\u04FF"

def homoglyph_attack(text, p, seed):
    rng = random.Random(seed)
    n_cyr = sum(_is_cyr(c) for c in text); n_lat = sum(c.isascii() and c.isalpha() for c in text)
    table = _CYR2LAT if n_cyr >= n_lat else _LAT2CYR
    chars = list(text); elig = [i for i, c in enumerate(chars) if c in table]
    for i in rng.sample(elig, int(round(p * len(elig)))):
        chars[i] = table[chars[i]]
    return "".join(chars)

def typo_attack(text, p, seed):
    rng = random.Random(seed)
    words = text.split(" ")
    alphabet = sorted({c for c in text if c.isalpha()}) or list("abcdefghijklmnopqrstuvwxyz")
    elig = [i for i, w in enumerate(words) if sum(c.isalpha() for c in w) >= 3]
    for i in rng.sample(elig, int(round(p * len(elig)))):
        w = list(words[i]); pos = [j for j, c in enumerate(w) if c.isalpha()]
        op = rng.choice(["swap", "delete", "dup", "sub"]); j = rng.choice(pos[:-1]) if op == "swap" else rng.choice(pos)
        if op == "swap":
            w[j], w[j + 1] = w[j + 1], w[j]
        elif op == "delete":
            del w[j]
        elif op == "dup":
            w.insert(j, w[j])
        else:
            w[j] = rng.choice(alphabet)
        words[i] = "".join(w)
    return " ".join(words)

def unicode_defense(text):
    """NFKC + strip zero-width + map look-alikes to each word's majority script."""
    t = unicodedata.normalize("NFKC", text).translate(_ZW)
    out = []
    for w in t.split(" "):
        n_cyr = sum(_is_cyr(c) for c in w); n_lat = sum(c.isascii() and c.isalpha() for c in w)
        if n_cyr and n_lat:
            table = _LAT2CYR if n_cyr >= n_lat else _CYR2LAT
            w = "".join(table.get(c, c) for c in w)
        out.append(w)
    return " ".join(out)

def doc_seed(doc_id, base=0):
    return int(stable_hash(f"{doc_id}|{base}", 8), 16)

def augment_train(train_df, kinds, seed):
    """Add one transformed copy per kind (T3/T4) to BOTH classes (never teach 'perturbation => machine')."""
    extra = []; rng = random.Random(seed)
    for k in kinds:
        fn = homoglyph_attack if k.startswith("T3") else typo_attack
        cp = train_df.copy()
        cp["text"] = [fn(t, rng.choice(CFG["attacks"][k]["levels"]), doc_seed(d, seed)) for t, d in zip(cp.text, cp.doc_id)]
        cp["doc_id"] = cp.doc_id + f"|aug{k[:2]}"
        extra.append(cp)
    return pd.concat([train_df] + extra, ignore_index=True)

## Language identification (GlotLID, lazy-loaded)

In [ ]:
_GLOTLID = None
def langid(texts):
    """Return list of (label, prob) with GlotLID (fastText). Label like 'kaz_Cyrl'.
    Uses the low-level predict to avoid the fastText/NumPy-2 'copy=False' incompatibility."""
    global _GLOTLID
    if _GLOTLID is None:
        import fasttext
        from huggingface_hub import hf_hub_download
        path = hf_hub_download(repo_id=CFG["tools"]["glotlid_repo"], filename="model.bin")
        _GLOTLID = fasttext.load_model(path)
    out = []
    for t in texts:
        pred = _GLOTLID.f.predict(t.replace("\n", " "), 1, 0.0, "strict")   # [(prob, label)]
        prob, label = pred[0] if pred else (0.0, "__label__und")
        out.append((label.replace("__label__", ""), float(prob)))
    return out

## Run registry & experiment naming
**Run ID** = `{exp}__{detector}__{train}__{heldout}__s{seed}__{cfghash}` e.g. `E1__mdeberta__kk__none__s13__a1b2c3d4`.
A run is *done* iff its predictions file exists **and** the registry says `done`. Notebooks skip done runs → no recomputation.

In [ ]:
REGISTRY = P["registry"] / "run_registry.csv"

def make_run_id(exp, detector, train, heldout, seed, extra: dict = None):
    payload = json.dumps({"exp": exp, "det": detector, "train": train, "held": heldout, "seed": seed,
                          "extra": extra or {}, "cfg": CFG["config_version"]}, sort_keys=True)
    return f"{exp}__{detector}__{train}__{heldout}__s{seed}__{stable_hash(payload)}"

def load_registry():
    if REGISTRY.exists():
        return pd.read_csv(REGISTRY)
    return pd.DataFrame(columns=["run_id", "exp", "detector", "train", "heldout", "seed", "status",
                                 "started_utc", "finished_utc", "seconds", "notes", "params_json"])

def register_run(run_id, status, **fields):
    """Insert/update one row of the run registry (atomic write)."""
    reg = load_registry()
    prev = reg[reg.run_id == run_id].to_dict("records")
    row = prev[0] if prev else {}
    row.update({"run_id": run_id, "status": status, **fields})
    if status == "running":
        row["started_utc"] = utcnow()
    if status in ("done", "failed"):
        row["finished_utc"] = utcnow()
    reg = pd.concat([reg[reg.run_id != run_id], pd.DataFrame([row])], ignore_index=True)
    tmp = Path(str(REGISTRY) + ".tmp"); reg.to_csv(tmp, index=False); os.replace(tmp, REGISTRY)

def pred_path(run_id):
    return P["predictions"] / f"{run_id}.parquet"

def is_done(run_id):
    reg = load_registry()
    ok = ((reg.run_id == run_id) & (reg.status == "done")).any()
    return bool(ok) and pred_path(run_id).exists()

## Data access helpers

In [ ]:
def load_dataset():
    return pd.read_parquet(P["processed"] / f"dataset_{CFG['data_version']}.parquet")

def load_eval_texts():
    """All texts any detector must score: val + test (clean), attacked test variants, external sets."""
    frames = []
    ds = load_dataset()
    frames.append(ds[ds.split.isin(["val", "test"])].assign(eval_set=lambda d: d.split, orig_doc_id=lambda d: d.doc_id))
    att = P["attacks"] / f"attacks_{CFG['data_version']}.parquet"
    if att.exists():
        frames.append(pd.read_parquet(att))
    ext = P["processed"] / f"external_{CFG['data_version']}.parquet"
    if ext.exists():
        frames.append(pd.read_parquet(ext))
    cols = ["doc_id", "orig_doc_id", "eval_set", "lang", "label", "family", "generator", "text"]
    df = pd.concat([f[[c for c in cols if c in f.columns]] for f in frames], ignore_index=True)
    assert df.doc_id.is_unique, "doc_id must be unique across evaluation sets"
    return df

def training_frame(ds, exp_spec):
    """Return (train_df, val_df) for an experiment spec:
       {'train_langs': [...], 'heldout_family': None|'openai'|..., 'augment': None|[...]}"""
    langs = exp_spec["train_langs"]; held = exp_spec.get("heldout_family")
    d = ds[ds.lang.isin(langs)]
    if held:
        held_topics = set(d[(d.label == 1) & (d.family == held)].pair_id)
        d = d[~d.pair_id.isin(held_topics)]          # drop machine texts of held family AND their paired human texts
    return d[d.split == "train"].reset_index(drop=True), d[d.split == "val"].reset_index(drop=True)

## Metrics

In [ ]:
from scipy.stats import rankdata

def auroc(y, s):
    """Mann–Whitney AUROC with average ranks (ties handled)."""
    y = np.asarray(y); s = np.asarray(s, dtype=float)
    n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def auroc_boot(y, s, idx):
    """Vectorised AUROC for a (B, n) matrix of resample indices."""
    y = np.asarray(y); s = np.asarray(s, dtype=float)
    Y = y[idx]; S = s[idx]
    R = rankdata(S, axis=1)
    n1 = Y.sum(1); n0 = Y.shape[1] - n1
    return ((R * Y).sum(1) - n1 * (n1 + 1) / 2) / (n1 * n0)

def threshold_at_fpr(y, s, fpr=0.01):
    """Smallest threshold whose FPR on (y, s) is <= fpr (higher score = machine)."""
    neg = np.sort(np.asarray(s)[np.asarray(y) == 0])[::-1]
    k = int(np.floor(fpr * len(neg)))
    return neg[k] + 1e-12 if k < len(neg) else -np.inf

def tpr_at_fpr(y, s, fpr=0.01):
    y = np.asarray(y); s = np.asarray(s)
    thr = threshold_at_fpr(y, s, fpr)
    return float((s[y == 1] > thr).mean()) if (y == 1).any() else np.nan

def best_f1_threshold(y, s):
    """Threshold maximising macro-F1 on a validation set."""
    from sklearn.metrics import f1_score
    cands = np.unique(np.quantile(s, np.linspace(0.01, 0.99, 99)))
    scores = [f1_score(y, (s > t).astype(int), average="macro") for t in cands]
    return float(cands[int(np.argmax(scores))])

def macro_f1(y, s, thr):
    from sklearn.metrics import f1_score
    return float(f1_score(y, (np.asarray(s) > thr).astype(int), average="macro"))

def ece(y, p, n_bins=10):
    """Expected calibration error for probabilities of the positive (machine) class."""
    y = np.asarray(y); p = np.clip(np.asarray(p, float), 0, 1)
    bins = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (p > lo) & (p <= hi) if lo > 0 else (p >= lo) & (p <= hi)
        if m.any():
            e += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(e)

def attack_success_rate(s_clean, s_attacked, thr):
    """Among machine texts detected before the attack, fraction no longer detected after it."""
    s_clean = np.asarray(s_clean); s_attacked = np.asarray(s_attacked)
    det = s_clean > thr
    return float((s_attacked[det] <= thr).mean()) if det.any() else np.nan

## Bootstrap & hypothesis tests

In [ ]:
def strat_boot_idx(y, B, seed=0):
    """Class-stratified bootstrap indices, shape (B, n)."""
    rng = np.random.default_rng(seed); y = np.asarray(y)
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    return np.hstack([rng.choice(pos, (B, len(pos))), rng.choice(neg, (B, len(neg)))])

def boot_ci_auroc(y, S, B=2000, seed=0, alpha=0.05):
    """S: array (n_seeds, n) of scores on the same docs. Hierarchical: resample docs, average AUROC over seeds."""
    S = np.atleast_2d(S); idx = strat_boot_idx(y, B, seed)
    boots = np.mean([auroc_boot(y, s, idx) for s in S], axis=0)
    point = float(np.mean([auroc(y, s) for s in S]))
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return point, float(lo), float(hi)

def paired_boot_delta(y, S1, S2, B=2000, seed=0, alpha=0.05):
    """ΔAUROC = A(S1) - A(S2) on the same docs; returns delta, CI, two-sided bootstrap p."""
    S1 = np.atleast_2d(S1); S2 = np.atleast_2d(S2); idx = strat_boot_idx(y, B, seed)
    b1 = np.mean([auroc_boot(y, s, idx) for s in S1], axis=0)
    b2 = np.mean([auroc_boot(y, s, idx) for s in S2], axis=0)
    d = b1 - b2
    point = float(np.mean([auroc(y, s) for s in S1]) - np.mean([auroc(y, s) for s in S2]))
    lo, hi = np.quantile(d, [alpha / 2, 1 - alpha / 2])
    p = float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
    return point, float(lo), float(hi), max(p, 1.0 / B)

def delong_test(y, s1, s2):
    """Paired DeLong test (Sun & Xu 2014 fast algorithm). Returns auc1, auc2, z, p."""
    from scipy.stats import norm
    y = np.asarray(y); order = np.argsort(-y, kind="stable")
    y = y[order]; X = np.vstack([np.asarray(s1)[order], np.asarray(s2)[order]])
    m = int(y.sum()); n = len(y) - m
    pos, neg = X[:, :m], X[:, m:]
    tx = np.array([rankdata(r) for r in pos]); ty = np.array([rankdata(r) for r in neg])
    tz = np.array([rankdata(r) for r in X])
    aucs = (tz[:, :m].sum(1) / m - (m + 1) / 2) / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(var) if var > 0 else 0.0
    return float(aucs[0]), float(aucs[1]), float(z), float(2 * norm.sf(abs(z)))

def mcnemar_test(y, pred1, pred2):
    """Exact McNemar test on correctness of two classifiers."""
    from scipy.stats import binomtest
    c1 = np.asarray(pred1) == np.asarray(y); c2 = np.asarray(pred2) == np.asarray(y)
    b = int((c1 & ~c2).sum()); c = int((~c1 & c2).sum())
    return b, c, (binomtest(b, b + c, 0.5).pvalue if b + c > 0 else 1.0)

def holm(pvals):
    """Holm–Bonferroni adjusted p-values (same order as input)."""
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def cliffs_delta_from_auc(a):
    """For two independent samples, Cliff's δ = 2·AUROC − 1."""
    return 2 * a - 1

print(f"[common] root={PROJECT_ROOT} | config v{CFG['config_version']} | data {CFG['data_version']} | langs={LANGS}")